Trabajo Práctico Grupal · Cámara de Diputados, 2022–2026

Puntos ideales por año, con una solapa por año y dos métodos lado a lado, a partir de `idps_py/idps.csv`
(generado por [`03_puntos_ideales.py`](03_puntos_ideales.py), que calcula los
puntos ideales con dos métodos: **PCA** (el atajo, primera componente
principal) y **Bayes** (el modelo de Clinton, Jackman y Rivers, implementado
en [`ideal_pscl.py`](ideal_pscl.py)).

Cada panel es un gráfico de barras divergentes: una fila por legislador,
ordenado de izquierda (azul, `idp < 0`) a derecha (rojo, `idp > 0`). En los
paneles de Bayes, una banda gris detrás de la barra muestra `idp_bayes ±
sd_bayes` — la incertidumbre de la estimación. Pasar el mouse sobre cualquier
barra muestra el nombre del legislador y su valor.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.special import log_ndtr

import igraph as ig

## Cargar los puntos ideales

In [ ]:
df = pd.read_csv("datos/idps.csv")
df.head()

## Orientar el PCA

El signo de `idp_pca` es arbitrario por año (lo decide sklearn en
`03_puntos_ideales.py`, no los datos — ver `pca_atajo()`). Lo reorientamos acá
contra `idp_bayes`, que sí viene fijado contra legisladores de referencia de
izquierda/derecha (`orientar()` en `03_puntos_ideales.py`): si la correlación
entre los dos métodos da negativa para un año, se invierte el signo de
`idp_pca` ese año.

In [ ]:
def orientar_pca(df):
    df = df.copy()
    for anio, g in df.groupby("anio"):
        r = g["idp_pca"].corr(g["idp_bayes"])
        if r is not None and r < 0:
            df.loc[g.index, "idp_pca"] = -df.loc[g.index, "idp_pca"]
    return df


df = orientar_pca(df)

## Puntos ideales por año: Bayes y PCA

In [ ]:
IZQUIERDA, DERECHA = "#2a78d6", "#e34948"
WHISKER, BASELINE, GRID = "#898781", "#c3c2b7", "#e1e0d9"
SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
TEXT, TEXT2, MUTED = "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, \'Segoe UI\', sans-serif"


def panel(fig, row, col, nombres, valores, sd=None):
    """Un panel: barras divergentes desde 0, ordenadas por valor, con
    opcionalmente un whisker +-sd (Bayes) y hover de nombre + idp."""
    orden = valores.argsort()
    v, nom = valores.to_numpy()[orden], nombres.to_numpy()[orden]
    colores = [DERECHA if x >= 0 else IZQUIERDA for x in v]
    kw = dict(
        x=v, y=list(range(len(v))), orientation="h",
        marker=dict(color=colores, line=dict(width=0)),
        showlegend=False,
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    )
    if sd is not None:
        s = sd.to_numpy()[orden]
        kw["error_x"] = dict(type="data", array=s, color=WHISKER, thickness=1, width=0)
        kw["customdata"] = list(zip(nom, s))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>idp = %{x:.2f} (\u00b1%{customdata[1]:.2f})<extra></extra>"
    else:
        kw["customdata"] = nom
        kw["hovertemplate"] = "<b>%{customdata}</b><br>idp = %{x:.2f}<extra></extra>"
    fig.add_trace(go.Bar(**kw), row=row, col=col)

In [ ]:
anios = sorted(df["anio"].unique())

bayes_domain = math.ceil((df["idp_bayes"].abs() + df["sd_bayes"]).max() * 10) / 10
pca_domain = 1.08

fig = make_subplots(
    rows=1, cols=2,
    column_titles=["Bayes · media posterior", "PCA · reescalado a [-1, 1], orientado contra Bayes"],
    horizontal_spacing=0.05,
)

for anio in anios:
    g = df[df["anio"] == anio]
    panel(fig, 1, 1, g["nombre"], g["idp_bayes"], sd=g["sd_bayes"])
    gp = g.dropna(subset=["idp_pca"])
    panel(fig, 1, 2, gp["nombre"], gp["idp_pca"])

# leyenda compartida: trazas invisibles, una sola vez
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=IZQUIERDA),
                          name="Izquierda (idp < 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                          marker=dict(size=10, color=DERECHA),
                          name="Derecha (idp > 0)"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=WHISKER, width=2),
                          name="\u00b1 1 desvio (incertidumbre, solo Bayes)"), row=1, col=1)


def visibles(anio):
    return [a == anio for a in anios for _ in range(2)] + [True] * 3


inicial = anios[-1]
for traza, visible in zip(fig.data, visibles(inicial)):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": visibles(a)}]) for a in anios]

fig.update_xaxes(range=[-bayes_domain, bayes_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=1)
fig.update_xaxes(range=[-pca_domain, pca_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=2)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, autorange="reversed")

fig.update_annotations(font=dict(size=13, color=TEXT2))

fig.update_layout(
    title=dict(text="Puntos ideales \u00b7 C\u00e1mara de Diputados, 2022\u20132026", x=0.01, y=0.98, yanchor="top",
               font=dict(size=20, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=900, width=1000,
    margin=dict(l=30, r=20, t=150, b=30),
    legend=dict(x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11)),
    bargap=0.1,
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.08, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig.show(config={"displaylogo": False, "scrollZoom": False})

PENDIENTE: Anotar cosas interesantes acá. Como el desvío estandar de Massa en el 2022 (equisdé)

PENDIENTE: IDEA - Inspirado en lo anterior, una tabla con los diputados con más desvío por año.. y cómo interpretamos eso en el mundo real

## Dot plot (Wilkinson): distribución por bloque

Otra forma de ver lo mismo: en vez de ordenar a cada legislador en su propia
fila (como arriba), se los apila — cada legislador es un círculo, y los que
caen en el mismo entorno de `idp_bayes` se amontonan uno sobre otro. El perfil
que arma la pila se lee como un histograma, pero sin perder la unidad
individual (sigue siendo "un legislador, un punto"). Es el **dot plot** de
Wilkinson (*"Dot Plots"*, 1999) — también se lo conoce como *dot histogram*.
No es lo mismo que un Cleveland dot plot (una marca por categoría, eje
categórico) ni que un beeswarm (jitter continuo, sin bins): acá el eje x está
discretizado en bins de ancho fijo y el apilamiento es vertical y exacto.

**Una aclaración sobre el color:** esto está coloreado por `bloque` real (el
bloque de cada legislador ese año, tomado de
`legisladores_bloque_provincia.csv`), no por una reconstrucción de los clusters/etiquetas que
se ven en la imagen de referencia — esa segmentación (p. ej. que UxP aparezca
partido en varios grupos separados) no se puede reconstruir de forma confiable
a partir de una captura de pantalla, así que en vez de inventar un criterio de
clustering fui con el dato real y verificable que tenemos: a qué bloque
pertenece cada uno.

In [ ]:
ANCHO_BIN = 0.06

bloques = pd.read_csv("datos/legisladores_bloque_provincia.csv")

dd = df.merge(bloques[["legislador", "yr", "bloque"]], left_on=["nombre", "anio"], right_on=["legislador", "yr"])
dd.groupby("anio")["bloque"].nunique()

Todos los bloques del Frente de Izquierda (PTS, Partido Obrero, Izquierda
Socialista, MST: 1 o 2 bancas cada uno, con nombres que cambian de un año a
otro) se agrupan bajo "Frente de Izquierda" —así se los suele nombrar en
conjunto— para que el color tenga sentido como categoría. El resto de los
bloques con menos de 10 legisladores *ese año* va a "Otros bloques". Evito reusar
azul/rojo acá: en los gráficos de arriba esos dos colores ya significan
izquierda/derecha (polaridad), y esto es una identidad categórica distinta
(bloque), así que mezclar los dos códigos de color confundiría más de lo que
aclara.

Entre años, el color sigue al bloque: Pro, UCR, LLA, etc. tienen el mismo color en
todas las solapas, y Unión por la Patria hereda el naranja de su antecesor, el
Frente de Todos. Como entre 2022 y 2026 pasan por la Cámara más bloques grandes
que colores distinguibles, los bloques que nunca coinciden en un mismo año
comparten color (por ejemplo Evolución Radical en 2022–2023 e Innovación Federal en
2025). La leyenda de cada solapa dice cuál es cuál.

In [ ]:
NOMBRE_BLOQUE = {
    "Frente De Todos": "Frente de Todos",
    "Union Por La Patria": "Unión por la Patria",
    "La Libertad Avanza": "La Libertad Avanza",
    "Pro": "PRO",
    "Ucr": "UCR",
    "Ucr - Union Civica Radical": "UCR",
    "Evolucion Radical": "Evolución Radical",
    "Coalicion Civica": "Coalición Cívica",
    "Encuentro Federal": "Encuentro Federal",
    "Democracia Para Siempre": "Democracia para Siempre",
    "Provincias Unidas": "Provincias Unidas",
    "Innovacion Federal": "Innovación Federal",
}

COLOR_BLOQUE = {
    "Frente de Izquierda": "#4a3aa7",
    "Frente de Todos": "#eb6834",
    "Unión por la Patria": "#eb6834",
    "PRO": "#e87ba4",
    "La Libertad Avanza": "#008300",
    "UCR": "#5c6bc0",
    "Provincias Unidas": "#1baf7a",
    "Coalición Cívica": "#1baf7a",
    "Encuentro Federal": "#1baf7a",
    "Innovación Federal": "#eda100",
    "Evolución Radical": "#eda100",
    "Democracia para Siempre": "#eda100",
    "Otros bloques": "#c3c2b7",
}

dd["bloque"] = dd["bloque"].where(~dd["bloque"].str.contains("izquierda", case=False), "Frente de Izquierda")
dd["bloque"] = dd["bloque"].replace(NOMBRE_BLOQUE)
grandes = dd.groupby(["anio", "bloque"])["nombre"].transform("size") >= 10
dd["bloque_mostrado"] = dd["bloque"].where(grandes | (dd["bloque"] == "Frente de Izquierda"), "Otros bloques")
assert dd["bloque_mostrado"].isin(COLOR_BLOQUE).all(), set(dd["bloque_mostrado"]) - set(COLOR_BLOQUE)
dd.groupby("anio")["bloque_mostrado"].value_counts().unstack(0).fillna(0).astype(int)

In [ ]:
def apilar(valores, ancho_bin):
    """Wilkinson: redondea cada valor al bin mas cercano y devuelve, para cada
    punto, en que nivel de la pila de ese bin le toca (0, 1, 2, ...)."""
    binned = (valores / ancho_bin).round() * ancho_bin
    nivel = binned.groupby(binned).cumcount()
    return binned, nivel


def mayor_hueco(g):
    gaps = g["idp_bayes"].diff()
    i_max = gaps.idxmax()
    antes, despues = g.loc[i_max - 1], g.loc[i_max]
    return gaps.loc[i_max], (antes["idp_bayes"] + despues["idp_bayes"]) / 2, antes["nombre"], despues["nombre"]


dots, huecos = {}, {}
for anio, g in dd.groupby("anio"):
    g = g.sort_values("idp_bayes").reset_index(drop=True)
    g["x_bin"], g["nivel"] = apilar(g["idp_bayes"], ANCHO_BIN)
    dots[anio], huecos[anio] = g, mayor_hueco(g)
    hueco, _, izq, der = huecos[anio]
    print(f"{anio}: mayor hueco {hueco:.2f}, entre {izq} y {der}")

In [ ]:
fig2 = go.Figure()

trazas_por_anio = {}
for anio, g in dots.items():
    trazas_por_anio[anio] = []
    for bloque_val, gb in g.groupby("bloque_mostrado"):
        trazas_por_anio[anio].append(len(fig2.data))
        fig2.add_trace(go.Scatter(
            x=gb["x_bin"], y=gb["nivel"],
            mode="markers",
            marker=dict(size=9, color=COLOR_BLOQUE[bloque_val], line=dict(width=0)),
            name=f"{bloque_val} ({len(gb)})",
            legendrank=1000 if bloque_val == "Otros bloques" else gb["idp_bayes"].median() * 100 + 500,
            customdata=list(zip(gb["nombre"], gb["bloque"], gb["idp_bayes"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{customdata[2]:.2f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))


def vista_anio(anio):
    hueco, x_hueco, _, _ = huecos[anio]
    visibles = [i in trazas_por_anio[anio] for i in range(len(fig2.data))]
    layout = dict(
        shapes=[dict(type="line", x0=x_hueco, x1=x_hueco, yref="paper", y0=0, y1=1,
                     line=dict(color=MUTED, width=1, dash="dot"))],
        annotations=[dict(x=x_hueco, y=0.85, yref="paper", text=f"mayor hueco: {hueco:.2f}", showarrow=False,
                          font=dict(size=11, color=TEXT2), xanchor="left", xshift=6)],
    )
    return visibles, layout


inicial = anios[-1]
visibles, layout_inicial = vista_anio(inicial)
for traza, visible in zip(fig2.data, visibles):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": vista_anio(a)[0]}, vista_anio(a)[1]])
           for a in anios]

x_max = math.ceil(dd["idp_bayes"].abs().max() * 10) / 10 + ANCHO_BIN
fig2.update_xaxes(range=[-x_max, x_max], zeroline=True, zerolinecolor=BASELINE, zerolinewidth=1,
                   gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig2.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

fig2.update_layout(
    **layout_inicial,
    title=dict(text="Puntos ideales (Bayes) por bloque \u00b7 C\u00e1mara de Diputados, 2022\u20132026",
               x=0.01, y=0.97, yanchor="top", font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=520, width=1000,
    margin=dict(l=40, r=20, t=110, b=40),
    legend=dict(orientation="h", y=-0.12, x=0),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig2.show(config={"displaylogo": False, "scrollZoom": False})

PENDIENTE - El hueco más grande en 0.71, qué significa eso en el mundo real? Que entre Grabois y Giordano se encuentra la brecha ideológica más grande y , por lo tanto, el camino más lejano para unir?

PENDIENTE - IDEA: Así como tenemos el espectro de diputados... Podríamos hacer un espectro de proyectos. Desde el 2022 al 2026, hacer un dotplot similar al de arriba pero ubicando los proyectos de acuerdo a que tan "de derecha" son o que tan "de izquierda" son. Esto se saca por la adhesión que tuvieron en la derecha/izquierda y rechazo que tuvieron en derecha/izquierda.

PENDIENTE - IDEA: Proyectos que no respetaron la polaridad derecha-izquierda. Es decir, proyectos que fueron muy divisivos pero tiene a una cantidad relevante de gente de izquierda votando igual que derecha. En clase nombraron la ley de aborto, por ejemplo, como un proyecto que tiene ejes o dimensiones que escapan al espectro derecha izquierda. A mi se me ocurre ley de presupuesto general (puede haber gente de izquierda a favor para evitar la discresionalidad del poder Ejecutivo VS gente de izquierda en contra porque avala el ajuste) 

## Mapa de proyectos: corte, discriminación y dirección

El modelo de Clinton, Jackman y Rivers no solo ubica legisladores: cada votación
*j* también tiene sus parámetros, en `P(Sí) = Φ(β_j · θ_i − α_j)`:

- **Corte** `α_j / β_j`: el punto del eje izquierda–derecha donde un legislador
  queda indiferente entre Sí y No. Cerca de 0 es una votación que parte la Cámara
  al medio; lejos de 0 es una votación donde solo un extremo quedó del otro lado
  (qué tan *extrema* es).
- **Discriminación** `|β_j|`: qué tan nítido es ese corte. Alta = el voto se
  explica casi del todo por la ideología (izquierda de un lado, derecha del
  otro); cerca de 0 = el eje izquierda–derecha no explica el voto (otros clivajes:
  territoriales, de conciencia, etc.).
- **Dirección** (color): el `idp_bayes` promedio de los que votaron **Sí**. Rojo =
  la apoyó la derecha; azul = la apoyó la izquierda. Es otra forma de leer el
  signo de β, en unidades de punto ideal.

`ideal_pscl.py` muestrea β y α pero no los guarda, así que acá se reestiman sin
volver a correr la cadena: para cada votación, un probit de los votos (Sí = 1,
No = 0, el resto afuera) contra los `idp_bayes` ya estimados, con el mismo prior
N(0, 25) del modelo (estimación MAP). El prior es también el que pone techo a β
en las votaciones con separación perfecta (≈ 11). Las votaciones unánimes quedan
afuera, igual que en `preparar_pscl()`.

In [ ]:
ses = pd.read_csv("datos/sesiones_dt.csv.gz", dtype={"legislador": str}, keep_default_na=False)
info = pd.read_csv("datos/info_sesiones_dt.csv")

votos = (ses.assign(y=ses["voto"].map({"AFIRMATIVO": 1.0, "NEGATIVO": 0.0}))
            .dropna(subset=["y"])
            .merge(df[["nombre", "anio", "idp_bayes"]], left_on=["legislador", "yr"], right_on=["nombre", "anio"]))


def parametros_votacion(g):
    x, y = g["idp_bayes"].to_numpy(), g["y"].to_numpy()
    if y.min() == y.max():
        return None
    s = 2 * y - 1
    menos_log_post = lambda p: -log_ndtr(s * (p[0] * x - p[1])).sum() + 0.02 * (p ** 2).sum()
    beta, alpha = minimize(menos_log_post, [0.0, 0.0], method="BFGS").x
    return pd.Series({"beta": beta, "alpha": alpha, "corte": alpha / beta,
                      "discriminacion": abs(beta), "idp_si": x[y == 1].mean(),
                      "afirmativos": int(y.sum()), "negativos": int(len(y) - y.sum())})


proyectos = (votos.groupby(["anio", "id_votacion"]).apply(parametros_votacion).dropna().reset_index()
                  .merge(info[["id_votacion", "titulo", "fecha", "resultado"]], on="id_votacion"))
proyectos.describe().round(2)

In [ ]:
RANGO_CORTE = 2.5

visibles = proyectos[proyectos["corte"].abs() <= RANGO_CORTE]
print(f"{len(proyectos) - len(visibles)} de {len(proyectos)} votaciones con |corte| > {RANGO_CORTE} quedan fuera "
      f"del gráfico (casi unánimes, con discriminación baja: el corte no está bien definido)")

fig3 = go.Figure()
for anio in anios:
    g = visibles[visibles["anio"] == anio]
    fig3.add_trace(go.Scatter(
        x=g["corte"], y=g["discriminacion"], mode="markers", name=str(anio),
        marker=dict(size=9, color=g["idp_si"], colorscale=[[0, IZQUIERDA], [0.5, GRID], [1, DERECHA]],
                    cmin=-1, cmax=1, cmid=0, line=dict(width=1, color=SURFACE),
                    colorbar=dict(title=dict(text="idp medio<br>de los Sí", font=dict(size=11, color=TEXT2)),
                                  tickvals=[-1, 0, 1], ticktext=["izquierda", "centro", "derecha"],
                                  tickfont=dict(size=10, color=MUTED), thickness=12, len=0.6)),
        customdata=list(zip(g["titulo"], g["fecha"], g["resultado"], g["idp_si"], g["afirmativos"], g["negativos"])),
        hovertemplate=("<b>%{customdata[0]}</b><br>%{customdata[1]} · %{customdata[2]} "
                       "(%{customdata[4]:.0f} Sí / %{customdata[5]:.0f} No)<br>"
                       "corte = %{x:.2f} · discriminación = %{y:.2f} · idp medio Sí = %{customdata[3]:.2f}<extra></extra>"),
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    ))

botones = [dict(label="Todos", method="update", args=[{"visible": [True] * len(anios)}])]
botones += [dict(label=str(a), method="update", args=[{"visible": [b == a for b in anios]}]) for a in anios]

fig3.update_xaxes(title=dict(text="corte (α/β) · ← izquierda · derecha →", font=dict(size=12, color=TEXT2)),
                  range=[-RANGO_CORTE, RANGO_CORTE], zeroline=True, zerolinecolor=BASELINE,
                  gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig3.update_yaxes(title=dict(text="discriminación |β|", font=dict(size=12, color=TEXT2)),
                  rangemode="tozero", gridcolor=GRID, zeroline=False, tickfont=dict(size=11, color=MUTED))
fig3.update_layout(
    title=dict(text="Votaciones 2022–2026: dónde cortan, qué tan nítido y quién las apoyó",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=600, width=1000, showlegend=False,
    margin=dict(l=60, r=20, t=110, b=60),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.01, xanchor="left",
                      y=1.02, yanchor="bottom", showactive=True, pad=dict(r=4, t=0),
                      bgcolor=SURFACE, bordercolor=GRID, font=dict(size=11, color=TEXT2))],
)
fig3.show(config={"displaylogo": False, "scrollZoom": False})

Cómo leerlo: arriba = votaciones puramente ideológicas (el voto se predice
casi perfecto con el punto ideal); abajo = votaciones que el eje
izquierda–derecha no explica — ahí hay que buscar los proyectos que "no
respetaron la polaridad" de la idea de arriba (aborto, presupuesto, etc.). El
corte en el eje x dice dónde parte la Cámara: cerca de 0, mitad y mitad; hacia
los costados, solo un extremo quedó del otro lado.

## Grafos de cercanía y comunidades

Lo mismo que `04_redes.py` (el script de redes de la materia), pero dibujado acá
con plotly, con una solapa por año. Se parte de `datos/distancias_idps.csv.gz`: el producto
cartesiano de diputados de cada año, con la distancia entre sus puntos ideales
(`|idp_bayes A - idp_bayes B|`).

Para cada año:

1. Se arma la red completa: un nodo por diputado y una arista por par, con la
   distancia como peso. Sobre esa red se calcula la **intermediación**
   (betweenness) de cada diputado.
2. Se **poda**: se borran las aristas con distancia mayor a `PUNTO_CORTE` (0.4),
   y después los diputados que quedan con menos de 3 uniones.
3. Se detectan **comunidades** con fast greedy (peso `1 / distancia`, así una
   unión corta pesa más), cortando donde la modularidad es máxima.

En el dibujo, el color es la comunidad (las 8 más grandes; el resto en gris), el
tamaño del nodo es su intermediación y se etiquetan los 5 diputados con más
intermediación (pasar el mouse sobre cualquier nodo muestra nombre, bloque e
intermediación). La leyenda dice, para cada comunidad, qué bloque es mayoría y en
qué proporción. La posición sale de Fruchterman-Reingold, así que la forma
cambia un poco entre corridas (la estructura no).

In [ ]:
PUNTO_CORTE = 0.4
COLORES_COMUNIDAD = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTRAS, ARISTA = "#c3c2b7", "#898781"

distancias = pd.read_csv("datos/distancias_idps.csv.gz").rename(
    columns={"nombre1": "Source", "nombre2": "Target", "distancia": "weight"})
distancias = distancias[distancias.Source < distancias.Target]


def armar_grafo(anio):
    dist = distancias[distancias.anio == anio]
    dips = bloques[bloques.yr == anio]
    g = ig.Graph.DataFrame(dist[["Source", "Target", "weight"]], directed=False,
                           vertices=dips[["legislador"]], use_vids=False)
    g.vs["bloque"] = list(dips.bloque)
    return g


def podar(g):
    g = g.copy()
    g.simplify(combine_edges="sum")
    g.delete_edges(g.es.select(weight_gt=PUNTO_CORTE))
    g.simplify(combine_edges="sum")
    g.delete_vertices(g.vs.select(_degree_lt=3))
    return g


def intermediacion(g):
    n = g.vcount()
    return [b * 2 / ((n - 1) * (n - 2)) for b in g.betweenness(directed=False, weights="weight")]


def comunidades(g):
    arbol = g.community_fastgreedy(weights=[1 / w for w in g.es["weight"]])
    membresia = ig.community_to_membership(arbol.merges, g.vcount(), g.vcount() - arbol.optimal_count)
    return ig.VertexClustering(g, membresia)

In [ ]:
def calcular_red(anio):
    g = armar_grafo(anio)
    g.vs["betweenness"] = intermediacion(g)
    g = podar(g)
    x = comunidades(g)
    por_tamanio = sorted(range(len(x)), key=lambda c: -len(x[c]))
    xs, ys = zip(*g.layout_fruchterman_reingold(weights=[1 / w for w in g.es["weight"]], niter=2000).coords)
    bloque = ["Frente de Izquierda" if "izquierda" in b.lower() else NOMBRE_BLOQUE.get(b, b) for b in g.vs["bloque"]]
    nodos = pd.DataFrame({"nombre": [n.replace('"', "").title() for n in g.vs["name"]], "bloque": bloque,
                          "betweenness": g.vs["betweenness"], "x": xs, "y": ys,
                          "comunidad": [por_tamanio.index(m) for m in x.membership]})
    aristas = [e.tuple for e in g.es]
    return nodos, aristas, len(x)


redes = {anio: calcular_red(anio) for anio in anios}

En 2025 hay más diputados que bancas (257) porque el 10 de diciembre cambió la
composición de la Cámara: en `legisladores_bloque_provincia.csv` aparecen tanto
los que salieron como los que entraron ese año.

In [ ]:
fig4 = go.Figure()
trazas_red = {}
titulos_red = {}
etiquetas_red = {}

for anio, (nodos, aristas, n_comunidades) in redes.items():
    trazas_red[anio] = []
    titulos_red[anio] = (f"Grafo de cercan\u00eda \u00b7 {anio} \u00b7 {len(nodos)} diputados, "
                         f"{len(aristas)} uniones, {n_comunidades} comunidades")

    ex, ey = [], []
    for a, b in aristas:
        ex += [nodos.x[a], nodos.x[b], None]
        ey += [nodos.y[a], nodos.y[b], None]
    trazas_red[anio].append(len(fig4.data))
    fig4.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color="rgba(137, 135, 129, 0.06)", width=0.5),
                                hoverinfo="skip", showlegend=False))

    nodos = nodos.assign(grupo=nodos["comunidad"].clip(upper=len(COLORES_COMUNIDAD)))
    for grupo, gn in nodos.groupby("grupo"):
        if grupo < len(COLORES_COMUNIDAD):
            bloque = gn["bloque"].mode()[0]
            color = COLORES_COMUNIDAD[grupo]
            nombre = f"{len(gn)} dip. \u00b7 {(gn['bloque'] == bloque).mean():.0%} {bloque}"
        else:
            color = OTRAS
            nombre = f"{len(gn)} dip. \u00b7 otras {n_comunidades - len(COLORES_COMUNIDAD)} comunidades"
        trazas_red[anio].append(len(fig4.data))
        fig4.add_trace(go.Scatter(
            x=gn["x"], y=gn["y"], mode="markers", name=nombre,
            marker=dict(size=(10 + 500 * gn["betweenness"]) ** 0.5 * 1.3, color=color,
                        line=dict(width=0.6, color=SURFACE)),
            customdata=list(zip(gn["nombre"], gn["bloque"], gn["betweenness"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>intermediaci\u00f3n = %{customdata[2]:.3f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))

    top = nodos.nlargest(5, "betweenness").sort_values("x")
    etiquetas_red[anio] = [dict(x=n.x, y=n.y, text=n.nombre, showarrow=True, arrowhead=0, arrowwidth=0.6,
                                arrowcolor=MUTED, ax=0, ay=-40 - 28 * k, font=dict(size=11, color=TEXT),
                                bgcolor=SURFACE, borderpad=1)
                           for k, n in enumerate(top.itertuples())]


def vista_red(anio):
    visibles = [i in trazas_red[anio] for i in range(len(fig4.data))]
    return [{"visible": visibles}, {"title.text": titulos_red[anio], "annotations": etiquetas_red[anio]}]


inicial = anios[-1]
for traza, visible in zip(fig4.data, vista_red(inicial)[0]["visible"]):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=vista_red(a)) for a in anios]

fig4.update_xaxes(visible=False)
fig4.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
fig4.update_layout(
    annotations=etiquetas_red[inicial],
    title=dict(text=titulos_red[inicial], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=750, width=1100,
    margin=dict(l=20, r=20, t=110, b=20),
    legend=dict(itemsizing="constant", title=dict(text="Comunidad \u00b7 bloque mayoritario", font=dict(size=11, color=TEXT2)),
                x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11, color=TEXT2)),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig4.show(config={"displaylogo": False, "scrollZoom": False})

PENDIENTE - IDEA: Les gusta la idea de una nube de palabras?? Teniendo los títulos de los proyectos, podemos hacer una nube de los proyectos con mayor adhesión derecha, y una nube con proyectos con mayor adhesion izquierda

PENDIENTE : Los graficos de redes quedaron horribles. Habrá que hacerlos en HTML o sino trabajar directamente con comunidades